In [2]:
import urllib.request

# Configuration
dataset_name = "ruoka"
base_url = f"https://cs.uef.fi/~himat/WebRank/dataset_12/dataset_12/{dataset_name}"
num_pages = 100

# Extract GT keywords and store in a set
gt_keywords = set()

print(f"Extracting GT keywords from {dataset_name} dataset...")
for i in range(num_pages):
    try:
        gt_url = f"{base_url}/{i}/GT.txt"
        with urllib.request.urlopen(gt_url, timeout=5) as response:
            gt_text = response.read().decode("utf-8-sig").strip()
            keywords = gt_text.lower().split()
            gt_keywords.update(keywords)
    except:
        pass

print(f"✓ Extracted {len(gt_keywords)} unique GT keywords")
print(f"\nFirst 20 keywords: {list(gt_keywords)[:20]}")

Extracting GT keywords from ruoka dataset...
✓ Extracted 346 unique GT keywords

First 20 keywords: ['rieska', 'voileipäkakku', 'paistisuikaleet', 'faustino', 'lehtikaali', 'lisukke', 'gluteenito', 'kurpitsa', 'joulu', 'sekavihannekset', 'kuohuviinit', 'kotilääkäri', 'munakkaat', 'pestot', 'teeleipä', 'riisipallot', 'frozen', 'nutella', 'leipä', 'siikafileet']


## Step 2: Apply Stemming

In [3]:
from nltk.stem.snowball import SnowballStemmer
import nltk

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Initialize Finnish stemmer (ruoka dataset is Finnish)
stemmer = SnowballStemmer('finnish')

# Apply stemming
stemmed_keywords = {}
changed_count = 0

for keyword in gt_keywords:
    stemmed = stemmer.stem(keyword)
    stemmed_keywords[keyword] = stemmed
    if keyword != stemmed:
        changed_count += 1

# Show only changed keywords
print("STEMMING RESULTS")
print("=" * 60)
print(f"Total keywords: {len(gt_keywords)}")
print(f"Keywords changed by stemming: {changed_count}")
print(f"Keywords unchanged: {len(gt_keywords) - changed_count}")
print(f"\nChanged keywords (Original → Stemmed):")
print("-" * 60)

changed_keywords = [(orig, stemmed) for orig, stemmed in stemmed_keywords.items() if orig != stemmed]
for i, (orig, stemmed) in enumerate(sorted(changed_keywords)[:30], 1):
    print(f"{i:3d}. {orig:20s} → {stemmed}")

if len(changed_keywords) > 30:
    print(f"... and {len(changed_keywords) - 30} more changed keywords")

STEMMING RESULTS
Total keywords: 346
Keywords changed by stemming: 266
Keywords unchanged: 80

Changed keywords (Original → Stemmed):
------------------------------------------------------------
  1. aamiaine             → aamia
  2. aasialainen          → aasialain
  3. afrika               → afrik
  4. ankanrintaa          → ankanrint
  5. anna                 → an
  6. annosfondi           → annosfond
  7. appelsiinikastiketta → appelsiinikastik
  8. arancini             → aranc
  9. arkiruoat            → arkiruoa
 10. basmati              → basmat
 11. bataatti             → bataat
 12. broileri             → broiler
 13. broilerin            → broiler
 14. broilerinliha        → broilerinlih
 15. broileriruoat        → broileriruoa
 16. buffet               → buf
 17. carpaccio            → carpacio
 18. chicken              → chick
 19. couscoussalaatti     → couscoussalaat
 20. curry                → cury
 21. curryt               → cury
 22. espanja              → espanj
 23. 

## Step 3: Apply Lemmatization

In [4]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Download required NLTK data for lemmatization
for package in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger']:
    try:
        nltk.data.find(f'corpora/{package}')
    except LookupError:
        try:
            nltk.download(package, quiet=True)
        except:
            pass

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Apply lemmatization (using noun as default POS)
lemmatized_keywords = {}
changed_count_lemma = 0

for keyword in gt_keywords:
    # Try lemmatizing as noun, verb, and adjective, pick the shortest
    lemma_noun = lemmatizer.lemmatize(keyword, pos='n')
    lemma_verb = lemmatizer.lemmatize(keyword, pos='v')
    lemma_adj = lemmatizer.lemmatize(keyword, pos='a')
    
    # Pick the shortest lemma (most reduced form)
    lemmatized = min([lemma_noun, lemma_verb, lemma_adj], key=len)
    lemmatized_keywords[keyword] = lemmatized
    
    if keyword != lemmatized:
        changed_count_lemma += 1

# Show only changed keywords
print("LEMMATIZATION RESULTS")
print("=" * 60)
print(f"Total keywords: {len(gt_keywords)}")
print(f"Keywords changed by lemmatization: {changed_count_lemma}")
print(f"Keywords unchanged: {len(gt_keywords) - changed_count_lemma}")
print(f"\nChanged keywords (Original → Lemmatized):")
print("-" * 60)

changed_keywords_lemma = [(orig, lemma) for orig, lemma in lemmatized_keywords.items() if orig != lemma]
for i, (orig, lemma) in enumerate(sorted(changed_keywords_lemma)[:30], 1):
    print(f"{i:3d}. {orig:20s} → {lemma}")

if len(changed_keywords_lemma) > 30:
    print(f"... and {len(changed_keywords_lemma) - 30} more changed keywords")

LEMMATIZATION RESULTS
Total keywords: 346
Keywords changed by lemmatization: 1
Keywords unchanged: 345

Changed keywords (Original → Lemmatized):
------------------------------------------------------------
  1. tapas                → tapa


## Step 4: Compare Stemming vs Lemmatization

In [5]:
# Compare stemming vs lemmatization
print("COMPARISON: STEMMING vs LEMMATIZATION")
print("=" * 70)
print(f"{'Metric':<40s} {'Stemming':>12s} {'Lemmatization':>15s}")
print("-" * 70)
print(f"{'Total keywords':<40s} {len(gt_keywords):>12d} {len(gt_keywords):>15d}")
print(f"{'Keywords changed':<40s} {changed_count:>12d} {changed_count_lemma:>15d}")
print(f"{'Keywords unchanged':<40s} {len(gt_keywords)-changed_count:>12d} {len(gt_keywords)-changed_count_lemma:>15d}")
print(f"{'Change rate (%)':<40s} {changed_count/len(gt_keywords)*100:>11.1f}% {changed_count_lemma/len(gt_keywords)*100:>14.1f}%")
print("=" * 70)

# Find keywords where stemming and lemmatization differ
different_results = []
for keyword in gt_keywords:
    if stemmed_keywords[keyword] != lemmatized_keywords[keyword]:
        different_results.append((keyword, stemmed_keywords[keyword], lemmatized_keywords[keyword]))

print(f"\nKeywords with different results (Stemming ≠ Lemmatization): {len(different_results)}")
if different_results:
    print("\nExamples (Original → Stemmed → Lemmatized):")
    print("-" * 70)
    for i, (orig, stem, lemma) in enumerate(sorted(different_results)[:20], 1):
        print(f"{i:3d}. {orig:15s} → {stem:15s} → {lemma}")
    
    if len(different_results) > 20:
        print(f"... and {len(different_results) - 20} more differences")

COMPARISON: STEMMING vs LEMMATIZATION
Metric                                       Stemming   Lemmatization
----------------------------------------------------------------------
Total keywords                                    346             346
Keywords changed                                  266               1
Keywords unchanged                                 80             345
Change rate (%)                                 76.9%            0.3%

Keywords with different results (Stemming ≠ Lemmatization): 267

Examples (Original → Stemmed → Lemmatized):
----------------------------------------------------------------------
  1. aamiaine        → aamia           → aamiaine
  2. aasialainen     → aasialain       → aasialainen
  3. afrika          → afrik           → afrika
  4. ankanrintaa     → ankanrint       → ankanrintaa
  5. anna            → an              → anna
  6. annosfondi      → annosfond       → annosfondi
  7. appelsiinikastiketta → appelsiinikastik → appelsiinik